In [2]:
from pylab import *
from math import pow
import numpy as np
import pandas as pd

基于矩阵分解的协同过滤

In [3]:
def matrix_factorization(score,P,Q,dims,iters=100,eta=0.001,lambda=0.01):
    result=[]
    for step in range(iters):
        for i in range(len(score)):
            for j in range(len(score[i])):
                if R[i][j]>0:
                    eij=R[i][j]-np.dot(P[i,:],Q[:,j])
                    for k in range(dims):
                        P[i][k]=P[i][k]+eta*(2*eij*Q[k][j]-lambda*P[i][k])
                        Q[k][j]=Q[k][j]+eta*(2*eij*P[i][k]-lambda*Q[k][j])
        eR=np.dot(P,Q)
        e=0
        for i in range(len(R)):
            for j in range(len(R[i])):
                if R[i][j]>0:
                    e=e+(R[i][j]-np.dot(P[i,:],Q[:,j]))**2
                    for k in range(dims):
                        e=e+(lambda/2)*((P[i][k])**2+(Q[k][j])**2)
        result.append(e)
        if e<0.001:
            break
    return result

In [248]:
user_song = pd.read_table('out_song.txt', sep=',',header=None)

In [249]:
user = set(user_song.iloc[:,0])
song = set(user_song.iloc[:,1])
user = list(user)
song = list(song)
m = len(user)
n = len(song)
score = np.zeros((m,n))
for i in range(m):
    for j in range(n):
        ind = user_song[(user_song.iloc[:,0]==user[i])&(user_song.iloc[:,1]==song[j])].index.tolist()
        if ind != []:
            score[i,j] = user_song.iloc[ind,2]
k=2
P=np.random.rand(m,k) 
Q=np.random.rand(n,k).T
nP,nQ,result=matrix_factorization(score,P,Q,k)

In [250]:
score_M=np.dot(nP,nQ.T)
print("经矩阵分解后的评分矩阵score_M为：\n",score_M)

经矩阵分解后的评分矩阵score_M为：
 [[  17.51288116   -2.00651734   32.69698037 ...    9.44146037
    18.99945619  -48.68420327]
 [  44.76107207   39.41620939   28.94544019 ...   39.14185059
    51.00227917   62.67978064]
 [  64.04030274   33.48535846   69.50442943 ...   48.2813672
    71.7140049    -6.54892553]
 ...
 [  62.64385483   57.92647189   37.12158042 ...   55.71067275
    71.52996938   99.32679235]
 [  77.70118803   45.74616586   78.0549946  ...   60.30518277
    87.29234992   13.5516304 ]
 [  25.24130864  -12.96420632   59.47758115 ...   10.21387956
    26.83180855 -112.47722488]]


In [251]:
song_top = sorted(enumerate(score_M[20,:]), key=lambda x: x[1],reverse=True)[0:10]
s1 = score[np.nonzero(score!=0)]
nn = sum(s1)
s2 = score_M[np.nonzero(score!=0)]
mse=sum((s1-s2)**2)/nn

In [255]:
print(song_top)
print(mse)

[(321, 957.0399979454056), (728, 640.6961556380215), (817, 594.5514722754283), (489, 573.570831904549), (234, 563.9213540398566), (440, 548.4482524176084), (268, 540.9889591709205), (808, 527.1785303218239), (307, 507.07947789343723), (1011, 497.15410794380693)]
0.20680429076138274


基于用户的协同过滤

In [276]:
def similar(score,in1, in2):
    #计算用户1与用户2的similarity
    user1_score = score[in1,:]
    user2_score = score[in2,:]
    diff = abs(user1_score-user2_score)
    for i in range(len(user1_score)):
        if user1_score[i]==0 or user2_score[i]==0:
            diff[i]=0
    dist = sum(abs(diff))
    sim = 1 / (dist + 1)
    return sim
def recommend(score, user, song, user1, k1=10, k2=10):
    #基于用户的协同过滤算法
    m=len(user)
    n=len(song)
    dists = [0]*m
    ind = user.index(user1)
    for i in range(m):
        if i != ind:
            dists[i] = similar(score, ind, i)
    #print(dists)
    top_users = sorted(enumerate(dists), key=lambda x: x[1], reverse=True)[0:k1]
    result = [0]*n
    sim_sum = 0
    score_m = 0
    for j in range(n):
        if score[ind,j]==0:
            for i in range(len(top_users)):
                current_ind = top_users[i][0]
                #print(current_ind)
                if score[current_ind,j]!=0:
                    score_m += score[current_ind,j] * top_users[i][1]
                    sim_sum += top_users[i][1]
        if (score_m!=0) & (sim_sum!=0):
            result[j] = score_m / sim_sum
    top_song = sorted(enumerate(result), key=lambda x: x[1], reverse=True)[0:k2]
    
    return top_users, result, top_song

In [284]:
user1=user[200]
aa,bb,cc=recommend(score,user,song,user1,k1=100,k2=10)